#### Prompt templates

#### 1. String PromptTemplate (default)

In [125]:
from langchain_openai import ChatOpenAI

In [126]:
model = ChatOpenAI(model_name='gpt-4o-mini')

In [127]:
prompt = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Large Language Models (LLMs) are the latest models used in NLP.
Their superior performance over smaller models has made them incredibly
useful for developers building NLP enabled applications. These models
can be accessed via Hugging Face's `transformers` library, via OpenAI
using the `openai` library, and via Cohere using the `cohere` library.

Question: Which libraries and model providers offer LLMs?

Answer: """

In [128]:
print(model.invoke(prompt).content)

The libraries and model providers that offer LLMs are Hugging Face's `transformers` library, OpenAI using the `openai` library, and Cohere using the `cohere` library.


In reality, we’re unlikely to `hardcode` the `context` and `user question`. We’d feed them in via a template — which is where Langchain’s `PromptTemplate` comes in.

In [129]:
from langchain import PromptTemplate

In [130]:
template = """
Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Large Language Models (LLMs) are the latest models used in NLP.
Their superior performance over smaller models has made them incredibly
useful for developers building NLP enabled applications. These models
can be accessed via Hugging Face's `transformers` library, via OpenAI
using the `openai` library, and via Cohere using the `cohere` library.

Question: {query}

Answer: 
"""

In [133]:
prompt_template = PromptTemplate(
    input_variables = ["query"],
    template        = template
)

With this, we can use the format method on our prompt_template to see the effect of passing a query to the template.

In [75]:
print(
    prompt_template.format(
        query = "Which libraries and model providers offer LLMs?"
    )
)


Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Large Language Models (LLMs) are the latest models used in NLP.
Their superior performance over smaller models has made them incredibly
useful for developers building NLP enabled applications. These models
can be accessed via Hugging Face's `transformers` library, via OpenAI
using the `openai` library, and via Cohere using the `cohere` library.

Question: Which libraries and model providers offer LLMs?

Answer: 



**we can pass the output of this directly into an LLM object**

In [26]:
response = model.invoke(
    prompt_template.format(
        query="Which libraries and model providers offer LLMs?"
    )
)
print(response.content)

LLMs are offered by Hugging Face via the `transformers` library, OpenAI using the `openai` library, and Cohere using the `cohere` library.


#### 2. F-String PromptTemplate

In [134]:
prompt = PromptTemplate(
    template       = "The answer is {answer}",
    input_variables= ["answer"],
    template_format= "f-string"
)

print(prompt.format(answer=42))  # Output: The answer is 42

The answer is 42


#### 3. Jinja2 PromptTemplate

In [135]:
prompt = PromptTemplate(
    template       = "Hello {{ name }}!",
    input_variables= ["name"],
    template_format= "jinja2"
)

print(prompt.format(name="Alice"))  # Output: Hello Alice!

Hello Alice!


#### 4. Few Shot Prompt Templates

Example 01

In [94]:
from langchain.prompts import FewShotPromptTemplate

- Create a list of dictionaries, each showing an input and its expected output.
- These are the “few-shot” examples that teach the LLM the pattern.

In [99]:
# Define example pairs
examples = [
    {"input": "cat", "output": "chat"},
    {"input": "dog", "output": "chien"}
]

- Define how each example should be formatted in the final prompt.
- Here, each example will look like:

> Input: cat
> 
> Output: chat

In [100]:
example_prompt = PromptTemplate(
    template       = "Input: {input}\nOutput: {output}",
    input_variables= ["input", "output"]
)

In [104]:
few_shot_prompt = FewShotPromptTemplate(
    examples       =examples,                        # The list of example pairs
    example_prompt =example_prompt,                  # How to format each example
    prefix         ="Translate English to French:",  # Text before the examples
    suffix         ="Input: {input}\nOutput:",       # Text after the examples, with a placeholder for the new input
    input_variables=["input"]                        # Variables to fill in the suffix
)

In [102]:
print(few_shot_prompt.format(input="bird"))

Translate English to French:

Input: cat
Output: chat

Input: dog
Output: chien

Input: bird
Output:


Example 02

In [79]:
prompt = """

The following is a conversation with an AI assistant.
The assistant is typically sarcastic and witty, producing creative 
and funny responses to the users questions. 

Here are some examples: 

User: What is the meaning of life?
AI: 

"""

In [80]:
model.temperature = 1.0  # increase creativity/randomness of output

In [81]:
response = model.invoke(prompt)
print(response.content)

Ah, the age-old question! The meaning of life is roughly 42... or perhaps just an endless quest for Wi-Fi and snacks. You know, the essentials!



In this case, we’re asking for something amusing, a joke, in return to our serious question. 

However, we get a serious response even with the temperature — which increases randomness/creativity — set to 1.0.

To help the model, we can give it a few examples of the type of answers we’d like:

In [82]:
prompt = """The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 

User: How are you?
AI: I can't complain but sometimes I still do.

User: What time is it?
AI: It's time to get a watch.

User: What is the meaning of life?
AI: """

In [83]:
response = model.invoke(prompt)
print(response.content)

To find that one missing sock from the laundry. It's the only mystery worth solving. But if you're looking for a deeper answer, maybe it's just about trying to avoid existential dread while binge-watching your favorite series.


With our examples reinforcing the instructions we passed in the prompt, we’re much more likely to get a more amusing response. We can then formalize this process with Langchain’s FewShotPromptTemplate

In [84]:
from langchain import FewShotPromptTemplate

In [85]:
# create our examples
examples = [
    {
        "query": "How are you?",
        "answer": "I can't complain but sometimes I still do."
    }, {
        "query": "What time is it?",
        "answer": "It's time to get a watch."
    }
]

In [87]:
# create a example template
example_template = """
User: {query}
AI: {answer}
"""

In [88]:
# create a prompt example from above template
example_prompt = PromptTemplate(
    input_variables= ["query", "answer"],
    template       = example_template
)


In [89]:
# now break our previous prompt into a prefix and suffix
# the prefix is our instructions
prefix = """The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 
"""

In [90]:
# and the suffix our user input and output indicator
suffix = """
User: {query}
AI: """

In [91]:
# now create the few shot prompt template
few_shot_prompt_template = FewShotPromptTemplate(
    examples       = examples,
    example_prompt = example_prompt,
    prefix         = prefix,
    suffix         = suffix,
    input_variables= ["query"],
    example_separator = "\n\n"
)

In [92]:
query = "What is the meaning of life?"

In [93]:
print(few_shot_prompt_template.format(query=query))

The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 



User: How are you?
AI: I can't complain but sometimes I still do.



User: What time is it?
AI: It's time to get a watch.



User: What is the meaning of life?
AI: 


#### Example 

In [41]:
import openai
from openai import OpenAI

In [42]:
client = OpenAI()

In [43]:
llm_model = "gpt-3.5-turbo"

In [44]:
def get_completion(prompt, model=llm_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model      = model,
        messages   = messages,
        temperature= 0, 
    )
    return response.choices[0].message.content

In [45]:
get_completion("What is 1+1?")

'1+1 equals 2.'

In [46]:
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help right now, matey!
"""

In [47]:
style = """American English in a calm and respectful tone"""

In [48]:
prompt = f"""Translate the text that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [49]:
response = get_completion(prompt)
response

"I am really frustrated that my blender lid flew off and splattered my kitchen walls with smoothie! And to make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help right now, friend!"

#### 5. ChatPromptTemplate 

- is designed for chat-based models (like OpenAI’s ChatGPT). It lets you build prompts that simulate multi-turn conversations, including system, human, and AI messages.

- How it works:
    - You define a sequence of messages, each with a role (system, human, ai).
    - You can use variables in the messages, which get filled in when formatting.
    - The template produces a structured prompt that chat models understand.

In [105]:
from langchain.prompts import ChatPromptTemplate

In [106]:
# Define a chat prompt with system and human messages
prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful assistant."),
            ("human", "{question}")
])

In [107]:
# Format the prompt with a specific question
formatted = prompt.format(question="What is the capital of France?")
print(formatted)

System: You are a helpful assistant.
Human: What is the capital of France?
